# 01 – Ingest files to bronze (Auto Loader)

Loads all Olist CSV tables from the landing volume into bronze Delta tables.

| | |
|---|---|
| **Source** | `/Volumes/<catalog>/landing/raw/olist/<table>/` |
| **Target** | `<catalog>.bronze.olist_<table>` |
| **Logic** | `src/ingestion/autoloader.py` |
| **Config** | `src/ingestion/config.yml` |

- **Incremental:** only files not processed before are ingested (Auto Loader checkpoints).
- **Idempotent:** re-running without new files adds 0 rows.
- **Parameters:** `catalog` (environment), `batch_id` (job run id; generated automatically when run manually).

In [0]:
import sys
import uuid
from pathlib import Path

# Make the repo root importable: this notebook lives in /notebooks, code lives in /src
repo_root = str(Path.cwd().parent)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.ingestion.autoloader import ingest_table, load_config

# Parameters (widgets) - a Databricks Job passes them in, manual runs use defaults
dbutils.widgets.text("catalog", "olist_dev", "Target catalog")
dbutils.widgets.text("batch_id", "", "Batch id (job run id)")

CATALOG = dbutils.widgets.get("catalog")
BATCH_ID = dbutils.widgets.get("batch_id") or f"manual-{uuid.uuid4().hex[:8]}"

cfg = load_config(CATALOG)
print(f"Catalog: {CATALOG} | Batch: {BATCH_ID} | Tables: {len(cfg['tables'])}")

In [0]:
results, errors = [], []

for table in cfg["tables"]:
    try:
        results.append(ingest_table(spark, table, cfg, CATALOG, BATCH_ID))
        print(f"OK      {table}")
    except Exception as e:
        # Collect errors and continue: one broken table must not block the others
        errors.append((table, str(e)))
        print(f"FAILED  {table}: {e}")

if results:
    display(spark.createDataFrame(results))

# Fail the run at the end so a Databricks Job is marked as failed
if errors:
    raise RuntimeError(f"Ingestion failed for {len(errors)} table(s): {[t for t, _ in errors]}")